In [ ]:
import json

path = "/kaggle/input/datasets/mhien01604/arxiv-doi/with_doi_1.jsonl"

rows = []

with open(path, "r", encoding="utf-8") as f:
    for line in f:
        rows.append(json.loads(line))

print(len(rows))
print(type(rows[0]))
print(rows[0].keys())
print(rows[0])

In [ ]:
for k, v in rows[0].items():
    print(k, type(v), str(v)[:200])

In [ ]:
count = 0

with open(path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            count += 1

print("Số record:", count)

In [ ]:
import json
import pandas as pd

ARXIV_PATH = "/kaggle/input/datasets/organizations/Cornell-University/arxiv/arxiv-metadata-oai-snapshot.json"
MY_PATH = "/kaggle/input/datasets/mhien01604/arxiv-doi/with_doi_1.jsonl"

# load id của bạn trước
my_df = pd.read_json(MY_PATH, lines=True)
my_ids = set(my_df["id"].astype(str))

rows = []

# đọc arxiv từng dòng, chỉ giữ paper match
with open(ARXIV_PATH, "r", encoding="utf-8") as f:
    for line in f:
        paper = json.loads(line)
        paper_id = str(paper.get("id"))

        if paper_id in my_ids:
            versions = paper.get("versions", [])
            created = versions[0].get("created") if versions else None
            year = pd.to_datetime(created, errors="coerce").year if created else None

            rows.append({
                "id": paper_id,
                "title": paper.get("title"),
                "published_year": year,
                "categories": paper.get("categories")
            })

result = pd.DataFrame(rows)

print("My dataset records:", len(my_df))
print("Matched papers:", len(result))

display(result.head())

result.to_csv("/kaggle/working/matched_arxiv_year.csv", index=False)

In [ ]:
count_2018 = (result["published_year"] <= 2018).sum()

print("Papers published <= 2018:", count_2018)

In [ ]:
old_papers = result[result["published_year"] <= 2018].copy()

print("Total old papers:", len(old_papers))

display(old_papers.head())

old_papers.to_csv("/kaggle/working/papers_le_2018.csv", index=False)

In [ ]:
import pandas as pd

WITH_DOI_PATH = "/kaggle/input/datasets/mhien01604/arxiv-doi/with_doi_1.jsonl"
OLD_PATH = "/kaggle/working/papers_le_2018.csv"

df = pd.read_json(WITH_DOI_PATH, lines=True)
old_papers = pd.read_csv(OLD_PATH)

merged = df.merge(
    old_papers[["id", "published_year", "categories"]],
    on="id",
    how="inner"
)

print("with_doi records:", len(df))
print("old_papers records:", len(old_papers))
print("merged <=2018 records:", len(merged))

display(merged.head())

merged.to_json(
    "/kaggle/working/with_doi_le_2018.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

In [ ]:
# Kiểm tra độ dài full_text
merged['text_length'] = merged['full_text'].astype(str).str.len()

# số từ
merged['word_count'] = merged['full_text'].astype(str).str.split().str.len()

print("=== TEXT LENGTH ===")
print(merged['text_length'].describe())

print("\n=== WORD COUNT ===")
print(merged['word_count'].describe())

print(f"\nTrung bình ký tự : {merged['text_length'].mean():,.0f}")
print(f"Max ký tự        : {merged['text_length'].max():,.0f}")

print(f"\nTrung bình số từ : {merged['word_count'].mean():,.0f}")
print(f"Max số từ        : {merged['word_count'].max():,.0f}")

In [ ]:
# Kiểm tra thử 3 papers xem format
for i in range(3):
    print(f"\n=== Paper {i} ===")
    print(df['full_text'].iloc[i][:500])
    print("...")

In [ ]:
import pandas as pd
import re

df = pd.read_json("/kaggle/working/with_doi_le_2018.jsonl", lines=True)
df['word_count'] = df['full_text'].str.split().str.len()

# 1. Null / empty
print("=== NULL CHECK ===")
print(df[["title", "abstract", "full_text"]].isnull().sum())
print(f"Empty full_text: {(df['full_text'].str.strip() == '').sum()}")

# 2. Độ dài bất thường
print("\n=== LENGTH OUTLIERS ===")
print(f"Quá ngắn  (< 500 từ):    {(df['word_count'] < 500).sum()}")
print(f"Quá dài   (> 50,000 từ): {(df['word_count'] > 50000).sum()}")

# 3. Noise patterns
print("\n=== NOISE PATTERNS ===")
# Tách các biểu thức regex có backslash ra khỏi f-string trước (PEP 701 chỉ
# cho phép backslash trong {} từ Python 3.12+ — tách ra để tương thích cả
# Python <3.12, không đổi ý nghĩa regex gốc).
n_latex = df['full_text'].str.contains(r'\\\\', regex=True).sum()
n_citation = df['full_text'].str.contains(r'\[[\d,\s]+\]', regex=True).sum()
n_url = df['full_text'].str.contains(r'http', regex=True).sum()
n_figure = df['full_text'].str.contains(r'[Ff]ig(ure)?\.?\s*\d', regex=True).sum()

print(f"Có LaTeX (\\\\):          {n_latex}")
print(f"Có citation [1,2]:       {n_citation}")
print(f"Có URL:                  {n_url}")
print(f"Có Figure caption:       {n_figure}")

# 4. Encoding issues
print("\n=== ENCODING ISSUES ===")
def has_encoding_issue(text):
    if not isinstance(text, str): return False
    non_ascii = sum(1 for c in text if ord(c) > 127)
    return non_ascii / len(text) > 0.05
print(f"Non-ASCII > 5%: {df['full_text'].apply(has_encoding_issue).sum()}")

# 5. Duplicate
print("\n=== DUPLICATES ===")
print(f"Duplicate full_text: {df['full_text'].duplicated().sum()}")
print(f"Duplicate title:     {df['title'].duplicated().sum()}")


In [ ]:
import pandas as pd
import random
import re

random.seed(42)

DATA_PATH = "/kaggle/working/with_doi_le_2018.jsonl"  # sửa path

df = pd.read_json(DATA_PATH, lines=True)

print("Total papers (before CS/IT filter):", len(df))

# Phạm vi đã chốt: chỉ CS/IT. Giữ paper có ít nhất 1 category cs.*
df = df[df["categories"].astype(str).str.contains(r"\bcs\.", regex=True)].reset_index(drop=True)
print("Total papers (cs.* only):", len(df))

# Drop THẬT các bản trùng (cell QA phía trên chỉ đếm 4 dup full_text / 6 dup title, không xóa)
before = len(df)
df = df.drop_duplicates(subset=["full_text"]).reset_index(drop=True)
df = df.drop_duplicates(subset=["title"]).reset_index(drop=True)
print(f"Dedup full_text/title: {before} -> {len(df)} ({before - len(df)} paper bị loại)")

print(df.columns)
print("Total papers (final):", len(df))
display(df.head())


In [ ]:
import pandas as pd
import random
import re
import nltk
from nltk.tokenize import sent_tokenize

random.seed(42)

for pkg, resource in [("punkt", "tokenizers/punkt"), ("punkt_tab", "tokenizers/punkt_tab")]:
    try:
        nltk.data.find(resource)
    except LookupError:
        nltk.download(pkg, quiet=True)


def clean_text(text):
    text = str(text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


# token bị coi là "toán/ký hiệu" nếu chứa số hoặc ký hiệu toán học phổ biến
_MATH_TOKEN_RE = re.compile(r"[0-9=+\-*/^_{}\\<>|~$]")


def is_prose_chunk(text, min_words=100, max_symbol_ratio=0.15, max_non_ascii_ratio=0.05):
    """Lọc đoạn không phải prose thật: công thức toán vỡ, OCR lỗi, quá ngắn (spec §3.2)."""
    if not text or not text[0].isupper():
        return False

    tokens = text.split()
    if len(tokens) < min_words:
        return False

    symbol_tokens = sum(1 for t in tokens if _MATH_TOKEN_RE.search(t))
    if symbol_tokens / len(tokens) > max_symbol_ratio:
        return False

    non_ascii = sum(1 for c in text if ord(c) > 127)
    if len(text) > 0 and non_ascii / len(text) > max_non_ascii_ratio:
        return False

    return True


def split_into_sentence_chunks(text, target_words=275, min_words=200, max_words=350):
    """Chunk theo câu (spec §3): gộp câu thành đoạn ~200-350 từ, không cắt giữa câu/công thức."""
    sentences = sent_tokenize(clean_text(text))
    chunks = []
    current = []
    current_wc = 0

    for sent in sentences:
        sent_wc = len(sent.split())

        if current and current_wc + sent_wc > max_words:
            chunks.append(" ".join(current))
            current = []
            current_wc = 0

        current.append(sent)
        current_wc += sent_wc

        if current_wc >= target_words:
            chunks.append(" ".join(current))
            current = []
            current_wc = 0

    if current and current_wc >= min_words:
        chunks.append(" ".join(current))

    return [c for c in chunks if is_prose_chunk(c, min_words=min_words)]


def sample_body_chunks(full_text, k=3):
    chunks = split_into_sentence_chunks(full_text)

    # Bỏ chunk đầu (thường dính phần đầu bài/tiêu đề lặp lại), giữ nguyên logic cũ
    if len(chunks) > 3:
        chunks = chunks[1:]

    if len(chunks) <= k:
        return chunks

    return random.sample(chunks, k)


samples = []

for _, row in df.iterrows():
    paper_id = row["id"]
    title = row.get("title")
    categories = row.get("categories")
    published_year = row.get("published_year")

    abstract = clean_text(row["abstract"])
    full_text = clean_text(row["full_text"])

    if len(abstract.split()) >= 80:
        samples.append({
            "paper_id": paper_id,
            "title": title,
            "categories": categories,
            "published_year": published_year,
            "abstract": abstract,
            "source": "abstract",
            "source_type": "abstract",
            "text": abstract,
            "word_count": len(abstract.split()),
            "label": 0,
        })

    for i, chunk in enumerate(sample_body_chunks(full_text, k=3)):
        samples.append({
            "paper_id": paper_id,
            "title": title,
            "categories": categories,
            "published_year": published_year,
            "abstract": abstract,
            "source": f"body_chunk_{i+1}",
            "source_type": "body",
            "text": chunk,
            "word_count": len(chunk.split()),
            "label": 0,
        })

human_df = pd.DataFrame(samples)

# Dedup theo text đã chunk — lớp bảo hiểm cuối, bổ sung cho dedup full_text/title ở cell trên
before_dedup = len(human_df)
human_df = human_df.drop_duplicates(subset=["text"]).reset_index(drop=True)
print(f"Dedup theo text (chunk-level): {before_dedup} -> {len(human_df)} ({before_dedup - len(human_df)} loại)")

print("Human samples:", len(human_df))
print(human_df["source_type"].value_counts())
display(human_df.head())

human_df.to_json(
    "/kaggle/working/human_samples_2018_clean.jsonl",
    orient="records",
    lines=True,
    force_ascii=False,
)
